# Notebook 07 — SHAP Explainability

**Goal:** Use SHAP (SHapley Additive exPlanations) to understand *why* models make specific predictions.

SHAP assigns each feature a contribution score for every prediction, based on cooperative game theory.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
from pathlib import Path

from src.utils import set_seed

set_seed(42)
sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

RESULTS = Path('../results')

## 1. Load Model & Data

In [ ]:
# Use LightGBM for SHAP (tree-based SHAP is fast and exact)
lgbm_model = joblib.load(RESULTS / 'models' / 'lightgbm_best.joblib')

df = pd.read_parquet('../data/processed/features.parquet')
feature_cols = [c for c in df.columns if not c.startswith('label')]

X = np.nan_to_num(df[feature_cols].values.astype(np.float32))
y = df['label'].values

# Use test set
test_start = int(len(X) * 0.85)
X_test = X[test_start:]
y_test = y[test_start:]

# Subsample for speed (SHAP on full test set can be slow)
n_shap = min(5000, len(X_test))
X_shap = X_test[:n_shap]

print(f'SHAP analysis on {n_shap} samples, {len(feature_cols)} features')

## 2. Compute SHAP Values

In [ ]:
# TreeExplainer is exact and fast for tree-based models
explainer = shap.TreeExplainer(lgbm_model)
shap_values = explainer.shap_values(X_shap)

# shap_values is a list of 3 arrays (one per class)
print(f'SHAP values computed.')
print(f'Shape per class: {shap_values[0].shape}')
print(f'Classes: Down (0), Stationary (1), Up (2)')

## 3. SHAP Summary Plot (Global Feature Importance)

In [ ]:
# Class 2 = Up (most interesting for trading)
fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(shap_values[2], X_shap, feature_names=feature_cols,
                  max_display=20, show=False)
plt.title('SHAP Summary — Class "Up" (Top 20 Features)', fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS / 'plots' / 'shap_summary_up.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# All classes
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
class_names = ['Down', 'Stationary', 'Up']

for i, (ax, name) in enumerate(zip(axes, class_names)):
    plt.sca(ax)
    shap.summary_plot(shap_values[i], X_shap, feature_names=feature_cols,
                      max_display=15, show=False)
    ax.set_title(f'Class: {name}', fontsize=13)

plt.tight_layout()
plt.savefig(RESULTS / 'plots' / 'shap_summary_all_classes.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. SHAP Bar Plot (Mean Absolute Importance)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap, feature_names=feature_cols,
                  plot_type='bar', max_display=20,
                  class_names=class_names, show=False)
plt.title('Mean |SHAP| by Feature (All Classes)', fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS / 'plots' / 'shap_bar_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. SHAP Dependence Plots (Key Features)

In [ ]:
key_features_idx = {
    'ofi': feature_cols.index('ofi'),
    'spread': feature_cols.index('spread'),
    'volume_delta': feature_cols.index('volume_delta'),
    'depth_imbalance_1': feature_cols.index('depth_imbalance_1'),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for ax, (feat_name, feat_idx) in zip(axes.flat, key_features_idx.items()):
    plt.sca(ax)
    shap.dependence_plot(feat_idx, shap_values[2], X_shap,
                        feature_names=feature_cols, show=False, ax=ax)
    ax.set_title(f'{feat_name} — SHAP Dependence (Up class)')

plt.tight_layout()
plt.savefig(RESULTS / 'plots' / 'shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Single Prediction Explanation (Waterfall)

In [ ]:
# Explain a single prediction
sample_idx = 42
pred = lgbm_model.predict(X_shap[sample_idx:sample_idx+1])[0]
actual = y_test[sample_idx]
print(f'Sample {sample_idx}: Predicted={pred} ({class_names[pred]}), Actual={actual} ({class_names[actual]})')

# Waterfall for the predicted class
shap_explanation = shap.Explanation(
    values=shap_values[pred][sample_idx],
    base_values=explainer.expected_value[pred],
    data=X_shap[sample_idx],
    feature_names=feature_cols
)

fig, ax = plt.subplots(figsize=(10, 8))
shap.waterfall_plot(shap_explanation, max_display=15, show=False)
plt.title(f'SHAP Waterfall — Sample {sample_idx} (Predicted: {class_names[pred]})', fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS / 'plots' / 'shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Ablation Study

In [ ]:
from sklearn.metrics import f1_score, accuracy_score
from src.data_loader import train_val_test_split
from src.models import build_lightgbm

# Feature groups for ablation
feature_groups = {
    'All Features': feature_cols,
    'Without OFI': [c for c in feature_cols if c != 'ofi'],
    'Without Spread': [c for c in feature_cols if 'spread' not in c],
    'Without Volume Delta': [c for c in feature_cols if c != 'volume_delta'],
    'Without Depth Imbalances': [c for c in feature_cols if 'depth_imbalance' not in c],
    'Without Volatility': [c for c in feature_cols if c not in ['volatility', 'vol_regime']],
    'Price Only (mid, spread)': ['mid_price', 'spread', 'weighted_mid', 'wmid_deviation', 'log_return'],
    'Volume Only': [c for c in feature_cols if 'vol' in c.lower() or 'delta' in c],
}

ablation_results = []
for group_name, cols in feature_groups.items():
    # Get indices
    col_indices = [feature_cols.index(c) for c in cols if c in feature_cols]
    if not col_indices:
        continue
    
    X_sub = X[:, col_indices]
    splits = train_val_test_split(X_sub, y, train_ratio=0.7, val_ratio=0.15)
    
    model = build_lightgbm()
    model.fit(splits['X_train'], splits['y_train'])
    preds = model.predict(splits['X_test'])
    
    acc = accuracy_score(splits['y_test'], preds)
    f1 = f1_score(splits['y_test'], preds, average='macro')
    
    ablation_results.append({
        'Feature Set': group_name,
        '# Features': len(col_indices),
        'Accuracy': round(acc, 4),
        'F1 (macro)': round(f1, 4),
    })
    print(f'{group_name:40s} | {len(col_indices):3d} feat | Acc={acc:.4f} | F1={f1:.4f}')

ablation_df = pd.DataFrame(ablation_results)
ablation_df.to_csv(RESULTS / 'tables' / 'ablation_study.csv', index=False)
print(f'\nSaved: results/tables/ablation_study.csv')

In [ ]:
# Ablation chart
fig, ax = plt.subplots(figsize=(12, 6))
colors_ab = ['#2ecc71'] + ['#e74c3c'] * (len(ablation_df) - 1)
colors_ab[0] = '#3498db'  # all features

ax.barh(ablation_df['Feature Set'], ablation_df['F1 (macro)'], color=colors_ab)
ax.set_xlabel('F1 Score (macro)')
ax.set_title('Ablation Study — Feature Group Impact on F1')
for i, v in enumerate(ablation_df['F1 (macro)']):
    ax.text(v + 0.002, i, f'{v:.4f}', va='center')

plt.tight_layout()
fig.savefig(RESULTS / 'plots' / 'ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

**Key insights from SHAP:**
- Order Flow Imbalance (OFI) is typically the most predictive feature
- Spread and depth imbalance at Level 1 are strong secondary signals
- Volatility regime helps segment model performance
- The ablation study quantifies the contribution of each feature group

These findings align with market microstructure theory: short-term price movements are driven primarily by order flow dynamics at the top of the book.